In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Set absolute path for new metastore
new_metastore_path = os.path.abspath('/apps/sandbox/metastore')
metastore_url = f"jdbc:derby:;databaseName={new_metastore_path};create=true"

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.sql.catalogImplementation", "hive")
    .set("spark.sql.catalog.spark_catalog.warehouse","s3a://warehouse/default")
    .set("spark.sql.warehouse.dir", "s3a://warehouse/default/spark")
    .set("javax.jdo.option.ConnectionURL", metastore_url)
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("09-liquid-clustering").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-86a5e48c-c756-4fdc-98b7-679233fff85e;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark-3.5.3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 192ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from local-m2-cache in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   arti

In [ ]:
%load_ext sql

In [ ]:
%sql spark

In [ ]:
%%bash

## Delete Existing Delta Table /invoices_cow
aws --endpoint-url http://minio.sandbox.net:9010 s3 ls s3://warehouse/default/deltalake/lc_ex1 --recursive



In [9]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/lc_ex1 --recursive

delete: s3://warehouse/default/deltalake/lc_ex1/_delta_log/00000000000000000000.crc
delete: s3://warehouse/default/deltalake/lc_ex1/_delta_log/00000000000000000000.json
delete: s3://warehouse/default/deltalake/lc_ex1/_delta_log/_commits/
delete: s3://warehouse/default/deltalake/lc_ex1/part-00000-807296fe-60ef-4ee4-8be8-c124ed27c5f8-c000.snappy.parquet
delete: s3://warehouse/default/deltalake/lc_ex1/part-00001-253c13cf-a18c-4987-ac7e-ddfeda12fdde-c000.snappy.parquet
delete: s3://warehouse/default/deltalake/lc_ex1/part-00002-b009ef88-ecec-4396-ae7b-4278301727a2-c000.snappy.parquet


In [4]:
df1 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet")
df2 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df3 = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_201_99457.parquet")

In [5]:
df_union = df1.union(df2).union(df3)
df_union = df_union.select("customer_id", "category", "price", "quantity", "invoice_date")

In [11]:
%%sql 

DROP TABLE IF EXISTS spark_catalog.deltalake.lc_ex1;


Running query in 'SparkSession'

++
||
++
++

In [12]:

#
df_union.write.mode("overwrite").partitionBy("invoice_date").format("delta").saveAsTable("spark_catalog.deltalake.lc_ex1")

In [13]:
%%sql

OPTIMIZE spark_catalog.deltalake.lc_ex1
ZORDER BY customer_id; 

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/lc_ex1,"Row(numFilesAdded=797, numFilesRemoved=985, filesAdded=Row(min=2111, max=2540, avg=2329.7779171894604, totalFiles=797, totalSize=1856833), filesRemoved=Row(min=1215, max=2540, avg=2121.9543147208124, totalFiles=985, totalSize=2090125), partitionsOptimized=797, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=985, size=2090125), inputNumCubes=0, mergedFiles=Row(num=985, size=2090125), numOutputCubes=797, mergedNumCubes=None), clusteringStats=None, numBins=797, numBatches=1, totalConsideredFiles=985, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1775031419923, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"


In [14]:
%%sql

SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex1; 

Running query in 'SparkSession'

1 rows affected.

Field 1
99457


In [20]:
%%sql 

CREATE TABLE IF NOT EXISTS spark_catalog.deltalake.lc_ex2
USING DELTA
SELECT * FROM spark_catalog.deltalake.lc_ex1
CLUSTER BY ("invoice_date", "customer_id")



Running query in 'SparkSession'

++
||
++
++

In [24]:
%%sql 

ALTER TABLE spark_catalog.deltalake.lc_ex2 CLUSTER BY ("invoice_date", "customer_id")

Running query in 'SparkSession'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:

[PARSE_SYNTAX_ERROR] Syntax error at or near 'CLUSTER'.(line 1, pos 43)

== SQL ==
ALTER TABLE spark_catalog.deltalake.lc_ex2 CLUSTER BY ("invoice_date", "customer_id")
-------------------------------------------^^^




In [26]:
%%sql 

OPTIMIZE spark_catalog.deltalake.lc_ex2

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/lc_ex2,"Row(numFilesAdded=0, numFilesRemoved=0, filesAdded=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), filesRemoved=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), partitionsOptimized=0, zOrderStats=None, clusteringStats=None, numBins=0, numBatches=1, totalConsideredFiles=1, totalFilesSkipped=1, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1775037921008, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"


In [ ]:
from delta.tables import *

DeltaTable.create(spark) \
  .tableName("spark_catalog.deltalake.lc_ex2") \
  .addColumn("col1", "STRING") \
  .addColumn("col2", "INT") \
  .clusterBy("col1", "col2") \
  .execute()

In [18]:

df_union.write.format("delta").mode("overwrite").clusterBy("invoice_date", "customer_id").saveAsTable("spark_catalog.deltalake.lc_ex2")

AttributeError: 'DataFrameWriter' object has no attribute 'clusterBy'

In [27]:
%%sql

SELECT count(*)
FROM FROM spark_catalog.deltalake.lc_ex2; 

Running query in 'SparkSession'

1 rows affected.

Field 1
99457


In [22]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) AS total_sales
    from spark_catalog.deltalake.lc_ex1
    WHERE (invoice_date BETWEEN '2021-01-01' AND '2023-12-31') AND 
        customer_id = 201
    GROUP BY category
    """
)

CPU times: user 0 ns, sys: 2.12 ms, total: 2.12 ms
Wall time: 46.5 ms


DataFrame[category: string, total_sales: double]

In [23]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) AS total_sales
    from spark_catalog.deltalake.lc_ex2
    WHERE (invoice_date BETWEEN '2021-01-01' AND '2023-12-31') AND 
        customer_id = 201
    GROUP BY category
    """
)

CPU times: user 685 μs, sys: 1.15 ms, total: 1.84 ms
Wall time: 24.5 ms


DataFrame[category: string, total_sales: double]